In [1]:
import os
from pathlib import Path

if Path.cwd().name == "examples":
    os.chdir(Path.cwd().parent)

from hybrid_rag import providers
from hybrid_rag.config import settings

In [2]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,  # 'force=True' penting di Jupyter agar menimpa handler root bawaan IPython
)


In [3]:
import asyncio
import functools
import logging
import math
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import httpx
import openai
from tenacity import before_sleep_log, retry, retry_if_exception, stop_after_attempt

logger = logging.getLogger("host.models")


def is_transient(exc: BaseException) -> bool:
    """True kalau mengirim ulang permintaan yang sama bisa berhasil."""
    if isinstance(exc, httpx.HTTPStatusError):  # jalur rerank
        return exc.response.status_code in (408, 429) or exc.response.status_code >= 500
    if isinstance(exc, (httpx.TransportError, TimeoutError)):  # TimeoutError: stream macet di langchain-openai
        return True
    if isinstance(exc, (openai.APIConnectionError, openai.APITimeoutError)):
        return True
    return isinstance(exc, openai.APIStatusError) and (exc.status_code in (408, 409, 429) or exc.status_code >= 500)

def seconds(value) -> float | None:
    try:
        delay = float(value)
    except (TypeError, ValueError):
        try:
            delay = (parsedate_to_datetime(str(value)).astimezone(timezone.utc) - datetime.now(timezone.utc)).total_seconds()
        except (TypeError, ValueError, IndexError, OverflowError):
            return None
    return max(0.0, float(math.ceil(delay)))


def wait_seconds(state) -> float:
    """`retry_after` dari provider (body JSON, lalu header) kalau ada; kalau tidak 1s, 2s, 4s, maksimal 8s."""
    exc = state.outcome.exception()
    body = getattr(exc, "body", None)
    headers = getattr(getattr(exc, "response", None), "headers", None)
    delay = seconds(body.get("retry_after")) if isinstance(body, dict) else None
    if delay is None and headers is not None:
        delay = seconds(headers.get("retry-after"))
    return delay if delay is not None else min(8.0, 2.0 ** (state.attempt_number - 1))

def with_retry(attempts):
    def decorate(func):
        def policy(*args):
            return retry(
                stop=stop_after_attempt(max(1, attempts(*args))),
                wait=wait_seconds,
                retry=retry_if_exception(is_transient),
                before_sleep=before_sleep_log(logger, logging.WARNING),
                reraise=True,
            )(func)

        if asyncio.iscoroutinefunction(func):
            @functools.wraps(func)
            async def awrapper(*args, **kwargs):
                return await policy(*args)(*args, **kwargs)
            return awrapper

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return policy(*args)(*args, **kwargs)
        return wrapper

    return decorate

per_instance = with_retry(lambda self, *_: self.retry_attempts)

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

class RetryingChat(ChatOpenAI):
    retry_attempts: int = 1
    @per_instance
    def _generate_with_cache(self, *args, **kwargs):
        return super()._generate_with_cache(*args, **kwargs)
    @per_instance
    async def _agenerate_with_cache(self, *args, **kwargs):
        return await super()._agenerate_with_cache(*args, **kwargs)

class RetryingEmbeddings(OpenAIEmbeddings):
    retry_attempts: int = 1
    @per_instance
    def embed_documents(self, *args, **kwargs):
        return super().embed_documents(*args, **kwargs)
    @per_instance
    async def aembed_documents(self, *args, **kwargs):
        return await super().aembed_documents(*args, **kwargs)
    @per_instance
    def embed_query(self, *args, **kwargs):
        return super().embed_query(*args, **kwargs)
    @per_instance
    async def aembed_query(self, *args, **kwargs):
        return await super().aembed_query(*args, **kwargs)


In [5]:

def get_llm(model_id, *, base_url, api_key, timeout, max_retries, max_tokens, **model_kwargs):
    return RetryingChat(
        model=model_id, base_url=base_url, api_key=api_key, timeout=timeout, max_tokens=max_tokens,
        max_retries=0, retry_attempts=max_retries, **model_kwargs,
    )

def get_vlm(model_id, *, base_url, api_key, timeout, max_retries, max_tokens=None):
    return get_llm(model_id, base_url=base_url, api_key=api_key, timeout=timeout, max_retries=max_retries, max_tokens=max_tokens)

def get_embeddings(model_id, *, base_url, api_key, timeout, max_retries):
    return RetryingEmbeddings(
        model=model_id, base_url=base_url, api_key=api_key, timeout=timeout,
        max_retries=0, retry_attempts=max_retries,
        model_kwargs={"encoding_format": "float"},
        check_embedding_ctx_length=False,
    )

def get_reranker(model_id, *, base_url, api_key, timeout, max_retries):
    @with_retry(lambda *_: max_retries)
    async def post(payload):
        async with httpx.AsyncClient(timeout=timeout) as client:
            response = await client.post(base_url, json=payload, headers={"Authorization": f"Bearer {api_key}"})
            response.raise_for_status()
            return response.json()

    async def rerank(query, documents, top_n=None):
        if top_n is not None:
            top_n = min(top_n, len(documents))
        data = await post({"model": model_id, "query": query, "documents": documents, "top_n": top_n})
        return data["results"]

    return rerank


# Daftarkan ke hybrid_rag

In [6]:
providers.configure(get_llm=get_llm, get_vlm=get_vlm, get_embeddings=get_embeddings, get_reranker=get_reranker)

# LLM

In [7]:
settings.require("extract")
llm = providers.get_llm(
    settings.extract_llm_model,
    base_url=settings.extract_llm_binding_host,
    api_key=settings.extract_llm_binding_api_key,
    timeout=settings.extract_llm_timeout,
    temperature=0.0,
)

# VLM

In [8]:
import base64
from langchain_core.messages import HumanMessage

settings.require("vlm")
vlm = providers.get_vlm(
    settings.vlm_model,
    base_url=settings.vlm_binding_host,
    api_key=settings.vlm_binding_api_key,
    timeout=settings.vlm_timeout,
)
# image = base64.b64encode(Path("examples/contoh.png").read_bytes()).decode()  # ganti dengan gambar Anda
# message = HumanMessage(content=[
#     {"type": "text", "text": "Jelaskan gambar ini."},
#     {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image}"}},
# ])
# (await vlm.ainvoke([message])).content

# EMBEDDINGS

In [9]:
settings.require("embedding")
embeddings = providers.get_embeddings(
    settings.embedding_model,
    base_url=settings.embedding_binding_host,
    api_key=settings.embedding_binding_api_key,
    timeout=settings.embedding_timeout,
)
# vectors = await embeddings.aembed_documents(["pasal 1", "pasal 2"])
# len(vectors), len(vectors[0])  # (2, EMBEDDING_DIM)

# RERANK

In [10]:
rerank = providers.get_reranker(
    settings.rerank_model,
    base_url=settings.rerank_binding_host,
    api_key=settings.rerank_binding_api_key,
    timeout=settings.rerank_timeout,
)
# await rerank(query="ibu kota Indonesia", documents=["Jakarta adalah ibu kota.", "Pisang itu buah."], top_n=2)

# Ingestion

Skema database dibuat sekali lewat `make db-init`. Lalu parse file dan tulis ke satu workspace.

In [11]:
from hybrid_rag.ingestion import parse_file, workspace_pool as write_pool
from pathlib import Path
path = Path("examples/test2.pdf")  
async with parse_file(path) as content_list:
    async with write_pool.lease("workspace_c") as rag:
        await rag.insert_content_list(content_list, file_path=path, doc_id="doc-2")

2026-10-13 10:38:35,908 [INFO] hybrid_rag.ingestion.pipeline: [parse] starting file=test2.pdf


[Textract] Mengecek DPI efektif setiap halaman (threshold=1)...
[Textract] Semua 22 halaman lolos cek DPI efektif.


2026-10-13 10:39:31,309 [INFO] hybrid_rag.ingestion.parser.pdf.async_amazon_textract.parse: [Textract] Job async 3aa8df7c1d33aff54ef70c6988aea5e789453577b44f2a20b7aaefd9993c4f82 dimulai (22 halaman)
2026-10-13 10:40:05,526 [INFO] hybrid_rag.ingestion.parser.pdf.async_amazon_textract.parse: [Textract] Job 3aa8df7c1d33aff54ef70c6988aea5e789453577b44f2a20b7aaefd9993c4f82 SUCCEEDED: 5072 blocks


[Textract] Formatted JSON disimpan ke: /tmp/rag_assets_os1famkj/formatted_parsing_result.json


2026-10-13 10:41:08,724 [INFO] hybrid_rag.storage.s3.objects: [S3] 32/32 figure diupload
2026-10-13 10:41:08,727 [INFO] hybrid_rag.ingestion.pipeline: [parse] prepared 427 items from file=test2.pdf
2026-10-13 10:41:08,729 [INFO] hybrid_rag.workspace_pool: [WorkspacePool] initializing instance for workspace: workspace_c
2026-10-13 10:41:09,225 [INFO] hybrid_rag.storage.postgres.vector: PostgreSQL table: hybrid_rag_vdb_entities_mjai_titan_embed_text_v2_0_1024d
2026-10-13 10:41:09,226 [INFO] hybrid_rag.storage.postgres.vector: PostgreSQL table: hybrid_rag_vdb_relations_mjai_titan_embed_text_v2_0_1024d
2026-10-13 10:41:09,227 [INFO] hybrid_rag.storage.postgres.vector: PostgreSQL table: hybrid_rag_vdb_chunks_mjai_titan_embed_text_v2_0_1024d
2026-10-13 10:41:09,229 [INFO] hybrid_rag.storage.postgres.connection: Database : retrieval_toolkit (ingestion pool)
2026-10-13 10:41:09,229 [INFO] hybrid_rag.storage.postgres.connection: PostgreSQL, Retry config: attempts=10, backoff=3.0s, backoff_max=3

2026-10-13 10:41:38,995 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-13 10:41:39,149 [INFO] hybrid_rag.core.utils:  == LLM cache == saving: default:extract:39150be0711842d07944af6bb56d7838
2026-10-13 10:41:40,793 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-13 10:41:40,898 [INFO] hybrid_rag.core.utils:  == LLM cache == saving: default:extract:592ce478fddbd212b07b21721cc41059
2026-10-13 10:41:43,289 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-13 10:41:43,395 [INFO] hybrid_rag.core.utils:  == LLM cache == saving: default:extract:588ad7b8037ddc57520e9edd81126f32
2026-10-13 10:41:45,798 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-13 10:41:45,865 [INFO] hybrid_rag.core.utils:  == LLM cache == saving: default:extract:adee2a78462ae1b571dda6d2d

# Retrieval

In [13]:
from hybrid_rag.retrieval import QueryParam, retrieve_context

result = await retrieve_context(
    "Apa judul dan tagline paparan Ekonomi Kreatif ini?",
    workspace="workspace_c",
    param=QueryParam(include_citations=True),
)
print(result.context)

2026-10-13 10:57:11,175 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-13 10:57:11,390 [INFO] hybrid_rag.core.utils:  == LLM cache == saving: mix:keywords:e4b70b6baa15d99de40ea37adaaf7717
2026-10-13 10:57:11,512 [INFO] hybrid_rag.retrieval.query: low_level_keywords kosong; memakai high_level_keywords untuk pencarian entitas
2026-10-13 10:57:12,752 [INFO] httpx2: HTTP Request: POST https://vertex.pisdev.my.id/v1/embeddings "HTTP/1.1 200 OK"
2026-10-13 10:57:12,812 [INFO] hybrid_rag.retrieval.search: Query nodes: Ekonomi Kreatif, Judul Paparan, Tagline (top_k:5, cosine:0.2)
2026-10-13 10:57:13,262 [INFO] hybrid_rag.retrieval.search: Local query: 5 entites, 20 relations
2026-10-13 10:57:13,264 [INFO] hybrid_rag.retrieval.search: Query edges: Ekonomi Kreatif, Judul Paparan, Tagline (top_k:5, cosine:0.2)
2026-10-13 10:57:14,344 [INFO] hybrid_rag.retrieval.search: Global query: 8 entites, 5 relations
2026-10-13 10:57:14,456 [INFO] 


Knowledge Graph (Entity) :

```json
{"entity": "Padang Creative City Forum", "type": "organization", "description": "Jejaring kreatif di Kota Padang yang tercatat dalam data ICCN dengan singkatan PCCF."}
{"entity": "Iccn", "type": "organization", "description": "ICCN adalah akronim yang merujuk pada beberapa entitas organisasi atau jejaring berbeda yang beroperasi di Indonesia. Berdasarkan deskripsi yang tersedia, terdapat empat entitas utama yang dapat diidentifikasi melalui singkatan ini.\n\nPertama, ICCN dapat merujuk pada Indonesia Creative Cities Network (atau Indonesia Creative City Network). Entitas ini adalah jejaring nasional yang mengkoordinasikan berbagai komunitas kreatif di tingkat kota dan kabupaten. Jaringan ini memiliki cakupan administratif yang luas, mencakup berbagai provinsi di Indonesia, termasuk 10 provinsi di wilayah Sumatera serta Provinsi Kepulauan Bangka Belitung per November 2025. Organisasi ini berfungsi sebagai inisiatif koordinasi antar-daerah di bidang k